In [1]:
import os
import re
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Tuple

import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds


# -----------------------------------------------------------------------------
# User parameters
# -----------------------------------------------------------------------------
S1_SAFE = "/home/stagiaire/Téléchargements/S1A_IW_GRDH_1SDV_20210724T090717_20210724T090742_038917_049794_C7F7.SAFE"
S2_REF_TIF = "s2_l2a_stack_12bands_10m_float32_buenos_aires.tif"
OUT_S1_TIF = "./s1_vv_vh_db_aligned_to_s2_buenos_aires.tif"

# SNAP GPT binary
GPT_BIN = "/opt/esa-snap/bin/gpt"

# DEM inside SNAP
DEM_NAME = "SRTM 1Sec HGT"

# Terrain correction pixel spacing (intermediate). Final grid will match S2 exactly.
TC_PIXEL_SPACING_M = 10.0

# SNAP uses a polygon in lon/lat for Subset (EPSG:4326)
AOI_DENSIFY_PTS = 21


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def read_ref_grid(ref_tif: str) -> Tuple[CRS, rasterio.Affine, int, int]:
    """
    Read CRS, transform, width and height from a reference raster.

    Parameters
    ----------
    ref_tif : str
        Path to reference GeoTIFF.

    Returns
    -------
    tuple
        (crs, transform, width, height)
    """
    with rasterio.open(ref_tif) as src:
        return src.crs, src.transform, src.width, src.height


def require_epsg(crs: CRS) -> int:
    """
    Extract EPSG code from CRS.

    Parameters
    ----------
    crs : CRS
        Raster CRS.

    Returns
    -------
    int
        EPSG code.

    Raises
    ------
    ValueError
        If EPSG code cannot be extracted.
    """
    epsg = crs.to_epsg()
    if epsg is None:
        raise ValueError(
            "Reference CRS has no EPSG code. Use a projected CRS with EPSG (e.g., UTM)."
        )
    return int(epsg)


def s2_bounds_to_wkt_4326(s2_ref_tif: str, densify_pts: int = 21) -> str:
    """
    Convert S2 raster bounds to a WKT polygon in EPSG:4326.

    SNAP Subset operator expects geoRegion as WKT in lon/lat.

    Parameters
    ----------
    s2_ref_tif : str
        Path to the S2 reference raster.
    densify_pts : int
        Densification points for transforming bounds (safer for curved edges).

    Returns
    -------
    str
        WKT polygon in EPSG:4326.
    """
    with rasterio.open(s2_ref_tif) as ref:
        b = ref.bounds
        src_crs = ref.crs

    minx, miny, maxx, maxy = transform_bounds(
        src_crs,
        CRS.from_epsg(4326),
        b.left,
        b.bottom,
        b.right,
        b.top,
        densify_pts=densify_pts,
    )

    # WKT uses "lon lat"
    return (
        f"POLYGON(({minx} {miny}, {minx} {maxy}, {maxx} {maxy}, "
        f"{maxx} {miny}, {minx} {miny}))"
    )


def write_snap_graph_snap13_subset(
    graph_path: str,
    s1_safe: str,
    out_tif: str,
    epsg_code: int,
    dem_name: str,
    pixel_spacing_m: float,
    aoi_wkt_4326: str,
) -> None:
    """
    Write a SNAP 13-compatible GPT graph that:
    - Applies orbit file
    - Removes thermal noise
    - Calibrates to Sigma0 (linear) for VV/VH
    - Terrain-corrects to target EPSG
    - Crops to the S2 footprint (Subset) BEFORE writing (prevents huge files)
    - Writes GeoTIFF

    Parameters
    ----------
    graph_path : str
        Path to output XML graph.
    s1_safe : str
        Sentinel-1 GRD SAFE directory.
    out_tif : str
        Output GeoTIFF path (SNAP intermediate, already cropped).
    epsg_code : int
        EPSG code matching S2 reference CRS.
    dem_name : str
        DEM name available in SNAP.
    pixel_spacing_m : float
        Pixel spacing for Terrain Correction (meters).
    aoi_wkt_4326 : str
        AOI polygon WKT in EPSG:4326 (lon/lat).
    """
    graph_xml = f"""<graph id="S1_GRD_RTC_Sigma0_Subset_SNAP13">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>{s1_safe}</file>
    </parameters>
  </node>

  <node id="Apply-Orbit-File">
    <operator>Apply-Orbit-File</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>true</continueOnFail>
    </parameters>
  </node>

  <node id="ThermalNoiseRemoval">
    <operator>ThermalNoiseRemoval</operator>
    <sources>
      <sourceProduct refid="Apply-Orbit-File"/>
    </sources>
    <parameters>
      <removeThermalNoise>true</removeThermalNoise>
    </parameters>
  </node>

  <node id="Calibration">
    <operator>Calibration</operator>
    <sources>
      <sourceProduct refid="ThermalNoiseRemoval"/>
    </sources>
    <parameters>
      <outputSigmaBand>true</outputSigmaBand>
      <selectedPolarisations>VV,VH</selectedPolarisations>
      <outputImageScaleInDb>false</outputImageScaleInDb>
    </parameters>
  </node>

  <node id="Terrain-Correction">
    <operator>Terrain-Correction</operator>
    <sources>
      <sourceProduct refid="Calibration"/>
    </sources>
    <parameters>
      <demName>{dem_name}</demName>
      <demResamplingMethod>BILINEAR_INTERPOLATION</demResamplingMethod>
      <imgResamplingMethod>BILINEAR_INTERPOLATION</imgResamplingMethod>
      <pixelSpacingInMeter>{pixel_spacing_m}</pixelSpacingInMeter>
      <mapProjection>EPSG:{epsg_code}</mapProjection>
      <alignToStandardGrid>true</alignToStandardGrid>
      <standardGridOriginX>0.0</standardGridOriginX>
      <standardGridOriginY>0.0</standardGridOriginY>
      <nodataValueAtSea>false</nodataValueAtSea>
      <saveSelectedSourceBand>true</saveSelectedSourceBand>
    </parameters>
  </node>

  <node id="Subset">
    <operator>Subset</operator>
    <sources>
      <sourceProduct refid="Terrain-Correction"/>
    </sources>
    <parameters>
      <geoRegion>{aoi_wkt_4326}</geoRegion>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Subset"/>
    </sources>
    <parameters>
      <file>{out_tif}</file>
      <formatName>GeoTIFF</formatName>
    </parameters>
  </node>
</graph>
"""
    with open(graph_path, "w", encoding="utf-8") as f:
        f.write(graph_xml)


def run_gpt(graph_path: str, gpt_bin: str) -> None:
    """
    Run SNAP GPT with a graph.

    Parameters
    ----------
    graph_path : str
        Path to SNAP graph XML.
    gpt_bin : str
        Path to SNAP gpt executable.

    Raises
    ------
    RuntimeError
        If GPT fails.
    """
    cmd = [gpt_bin, graph_path, "-q", "4"]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "SNAP GPT failed.\n"
            f"Command: {' '.join(cmd)}\n\n"
            f"STDOUT:\n{proc.stdout}\n\n"
            f"STDERR:\n{proc.stderr}\n"
        )


def find_vv_vh_indices(src: rasterio.io.DatasetReader) -> Tuple[int, int]:
    """
    Find VV and VH band indices using descriptions; fallback to (1, 2).
    """
    desc = src.descriptions or ()
    vv_idx = None
    vh_idx = None

    for i, d in enumerate(desc, start=1):
        if not d:
            continue
        d_u = d.upper()
        if re.search(r"\bVV\b", d_u):
            vv_idx = i
        if re.search(r"\bVH\b", d_u):
            vh_idx = i

    if vv_idx is not None and vh_idx is not None:
        return vv_idx, vh_idx

    return 1, 2


def to_db_sigma0(sigma0_lin: np.ndarray) -> np.ndarray:
    """
    Convert linear sigma0 to dB: 10 * log10(sigma0).
    """
    x = sigma0_lin.astype(np.float32, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(x) & (x > 0.0)
    out[valid] = 10.0 * np.log10(x[valid])
    return out


def warp_to_ref_grid(
    arr: np.ndarray,
    src_crs: CRS,
    src_transform: rasterio.Affine,
    ref_crs: CRS,
    ref_transform: rasterio.Affine,
    ref_width: int,
    ref_height: int,
) -> np.ndarray:
    """
    Reproject an array onto an exact reference grid.
    """
    dst = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
    reproject(
        source=arr,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=np.nan,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )
    return dst


def write_vv_vh_stack(
    out_tif: str,
    vv_db: np.ndarray,
    vh_db: np.ndarray,
    ref_crs: CRS,
    ref_transform: rasterio.Affine,
) -> str:
    """
    Write a 2-band GeoTIFF: Band 1 = VV_dB, Band 2 = VH_dB.
    """
    out_dir = os.path.dirname(out_tif)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    height, width = vv_db.shape
    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 2,
        "dtype": "float32",
        "crs": ref_crs,
        "transform": ref_transform,
        "nodata": np.nan,
        "interleave": "band",
        "tiled": True,
        "compress": "deflate",
        "predictor": 3,
    }

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(vv_db, 1)
        dst.write(vh_db, 2)
        dst.set_band_description(1, "VV_dB")
        dst.set_band_description(2, "VH_dB")

    return out_tif


# -----------------------------------------------------------------------------
# Pipeline
# -----------------------------------------------------------------------------
ref_crs, ref_transform, ref_width, ref_height = read_ref_grid(S2_REF_TIF)
ref_epsg = require_epsg(ref_crs)

# Your S2 patch is 11x11 pixels, so the AOI is tiny; this prevents huge SNAP outputs.
aoi_wkt = s2_bounds_to_wkt_4326(S2_REF_TIF, densify_pts=AOI_DENSIFY_PTS)

tmp_dir = tempfile.mkdtemp(prefix="snap_s1_rtc_")
try:
    tmp_graph = str(Path(tmp_dir) / "s1_rtc_graph.xml")
    tmp_tif = str(Path(tmp_dir) / "s1_sigma0_tc_linear_subset.tif")

    write_snap_graph_snap13_subset(
        graph_path=tmp_graph,
        s1_safe=S1_SAFE,
        out_tif=tmp_tif,
        epsg_code=ref_epsg,
        dem_name=DEM_NAME,
        pixel_spacing_m=TC_PIXEL_SPACING_M,
        aoi_wkt_4326=aoi_wkt,
    )

    run_gpt(tmp_graph, GPT_BIN)

    with rasterio.open(tmp_tif) as src:
        vv_idx, vh_idx = find_vv_vh_indices(src)

        vv_lin = src.read(vv_idx).astype(np.float32)
        vh_lin = src.read(vh_idx).astype(np.float32)

        nodata = src.nodata
        if nodata is not None:
            vv_lin = np.where(vv_lin == nodata, np.nan, vv_lin)
            vh_lin = np.where(vh_lin == nodata, np.nan, vh_lin)

        vv_db = to_db_sigma0(vv_lin)
        vh_db = to_db_sigma0(vh_lin)

        # Force exact alignment to the S2 grid (same CRS/transform/shape).
        vv_db_aligned = warp_to_ref_grid(
            arr=vv_db,
            src_crs=src.crs,
            src_transform=src.transform,
            ref_crs=ref_crs,
            ref_transform=ref_transform,
            ref_width=ref_width,
            ref_height=ref_height,
        )

        vh_db_aligned = warp_to_ref_grid(
            arr=vh_db,
            src_crs=src.crs,
            src_transform=src.transform,
            ref_crs=ref_crs,
            ref_transform=ref_transform,
            ref_width=ref_width,
            ref_height=ref_height,
        )

    out_path = write_vv_vh_stack(
        out_tif=OUT_S1_TIF,
        vv_db=vv_db_aligned,
        vh_db=vh_db_aligned,
        ref_crs=ref_crs,
        ref_transform=ref_transform,
    )

finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

out_path


'./s1_vv_vh_db_aligned_to_s2_buenos_aires.tif'